# Num Rollouts Ablation Analysis

Analyze sampled-controller rollout-count ablations produced by `run_scripts/run_m3cot_validation_num_rollouts_ablation.sh` and `lvar_scripts/infer_lvar_m3cot_rollouts.py` for `G = 1, 2, 4, 8, 16, 32`.

The notebook compares raw rollout accuracy, best-of-N majority-vote accuracy, oracle accuracy, random-rollout accuracy, answer diversity, entropy, and simple efficiency metrics as the number of sampled rollouts changes.

In [ ]:
%matplotlib inline

from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent

# Point this at one ablation directory, an output root, or leave it recursive.
RUN_ROOT = ROOT / "outputs" / "inference" / "validation_num_rollouts_ablation"
EXPECTED_ROLLOUTS = [1, 2, 4, 8, 16, 32]

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)

## Load Results

Each run writes a summary JSON, raw rollout JSONL, accuracy-variant JSONL, and entropy sidecar JSON. The loader discovers summaries and follows paths recorded inside each summary when available.

In [ ]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} line {line_number}: {exc}") from exc
    return rows


def resolve_output_path(path_value):
    path = Path(path_value or "")
    if not path.is_absolute():
        path = ROOT / path
    return path


def discover_summaries(root):
    root = Path(root)
    if root.is_file() and root.name.endswith("_summary.json"):
        return [root]
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*_summary.json") if "rollout" in path.name)


summary_paths = discover_summaries(RUN_ROOT)
print(f"Discovered {len(summary_paths)} summary files under {RUN_ROOT}")
for path in summary_paths[:12]:
    print(" -", path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)

In [ ]:
summaries = []
rollout_rows = []
variant_rows = []
entropy_rows = []

for summary_path in summary_paths:
    summary = load_json(summary_path)
    run_id = str(summary_path.parent.relative_to(ROOT)) if summary_path.parent.is_relative_to(ROOT) else str(summary_path.parent)
    metrics = summary.get("metrics", {})
    num_rollouts = summary.get("num_rollouts")
    summaries.append({
        "run_id": run_id,
        "summary_path": str(summary_path),
        "dataset_partition": summary.get("dataset_partition"),
        "num_examples": summary.get("num_examples"),
        "num_rollouts": num_rollouts,
        "controller_temperature": summary.get("controller_temperature"),
        "seed": summary.get("seed"),
        "coarse_context": summary.get("coarse_context"),
        **metrics,
    })

    rollout_path = resolve_output_path(summary.get("rollout_predictions_path"))
    variant_path = resolve_output_path(summary.get("accuracy_variants_path"))
    entropy_path = resolve_output_path(summary.get("entropy_tracking_path"))

    for row in load_jsonl(rollout_path):
        row["run_id"] = run_id
        row["num_rollouts"] = num_rollouts
        rollout_rows.append(row)
    for row in load_jsonl(variant_path):
        row["run_id"] = run_id
        row["num_rollouts"] = num_rollouts
        variant_rows.append(row)
    for row in load_json(entropy_path) if entropy_path.exists() else []:
        row["run_id"] = run_id
        row["num_rollouts"] = num_rollouts
        entropy_rows.append(row)

summary_df = pd.DataFrame(summaries)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["num_rollouts", "controller_temperature", "seed"])
rollouts_df = pd.DataFrame(rollout_rows)
variants_df = pd.DataFrame(variant_rows)
entropy_df = pd.DataFrame(entropy_rows)

print(f"Summary rows: {len(summary_df):,}")
print(f"Rollout rows: {len(rollouts_df):,}")
print(f"Variant rows: {len(variants_df):,}")
print(f"Entropy rows: {len(entropy_df):,}")

if not summary_df.empty:
    observed = sorted(summary_df["num_rollouts"].dropna().astype(int).unique().tolist())
    missing = sorted(set(EXPECTED_ROLLOUTS) - set(observed))
    print("Observed rollout counts:", observed)
    if missing:
        print("Missing expected rollout counts:", missing)

## Accuracy By Rollout Count

Best-of-N and oracle accuracy should usually be the main curves for this ablation. Raw rollout accuracy is computed over all sampled trajectories, so it answers a different question.

In [ ]:
accuracy_cols = ["rollout_accuracy", "best_of_n_accuracy", "oracle_accuracy", "random_accuracy"]

if summary_df.empty:
    display(summary_df)
else:
    display_cols = [
        "run_id", "dataset_partition", "num_examples", "num_rollouts",
        "controller_temperature", "seed", "coarse_context", *accuracy_cols,
    ]
    table = summary_df[display_cols].sort_values("num_rollouts")
    display(table.style.format({col: "{:.2%}" for col in accuracy_cols}))

    plot_df = table.melt(
        id_vars=["num_rollouts"],
        value_vars=accuracy_cols,
        var_name="metric",
        value_name="accuracy",
    )
    plt.figure(figsize=(9, 5))
    if HAS_SEABORN:
        sns.lineplot(data=plot_df, x="num_rollouts", y="accuracy", hue="metric", marker="o")
    else:
        for metric, group in plot_df.groupby("metric"):
            plt.plot(group["num_rollouts"], group["accuracy"], marker="o", label=metric)
        plt.legend()
    plt.xscale("log", base=2)
    plt.xticks(EXPECTED_ROLLOUTS, EXPECTED_ROLLOUTS)
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.xlabel("Number of rollouts per prompt")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Number of Sampled Controller Rollouts")
    plt.tight_layout()
    plt.show()

## Marginal Returns

This view estimates how much each doubling of rollout count buys for best-of-N and oracle accuracy.

In [ ]:
if summary_df.empty:
    display(summary_df)
else:
    marginal = summary_df.sort_values("num_rollouts").copy()
    for col in ["best_of_n_accuracy", "oracle_accuracy", "random_accuracy"]:
        marginal[f"{col}_delta"] = marginal[col].diff()
        marginal[f"{col}_per_extra_rollout"] = marginal[f"{col}_delta"] / marginal["num_rollouts"].diff()
    display_cols = [
        "num_rollouts", "best_of_n_accuracy", "best_of_n_accuracy_delta", "best_of_n_accuracy_per_extra_rollout",
        "oracle_accuracy", "oracle_accuracy_delta", "oracle_accuracy_per_extra_rollout",
    ]
    display(marginal[display_cols].style.format({col: "{:.2%}" for col in display_cols if col != "num_rollouts"}))

    gain_df = marginal.melt(
        id_vars=["num_rollouts"],
        value_vars=["best_of_n_accuracy_delta", "oracle_accuracy_delta"],
        var_name="metric",
        value_name="accuracy_gain",
    ).dropna()
    if not gain_df.empty:
        plt.figure(figsize=(8, 4))
        if HAS_SEABORN:
            sns.barplot(data=gain_df, x="num_rollouts", y="accuracy_gain", hue="metric")
        else:
            for metric, group in gain_df.groupby("metric"):
                plt.plot(group["num_rollouts"], group["accuracy_gain"], marker="o", label=metric)
            plt.legend()
        plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
        plt.xlabel("New rollout count after doubling")
        plt.ylabel("Accuracy gain")
        plt.title("Marginal Accuracy Gain From More Rollouts")
        plt.tight_layout()
        plt.show()

## Diversity And Selection Behavior

Answer diversity indicates whether extra sampled trajectories create genuinely different final answers or mostly duplicate the same answer.

In [ ]:
if rollouts_df.empty:
    display(rollouts_df)
else:
    rollouts_df["correct"] = rollouts_df["correct"].astype(bool)
    per_prompt = rollouts_df.groupby(["num_rollouts", "run_id", "example_id"]).agg(
        rollout_count=("rollout_idx", "count"),
        any_correct=("correct", "any"),
        random_accuracy_estimate=("correct", "mean"),
        unique_answers=("answer_key", "nunique"),
        majority_fraction=("answer_key", lambda values: Counter(values).most_common(1)[0][1] / len(values)),
        mean_decoded_entropy=("token_entropy_mean", "mean"),
        mean_hidden_entropy=("hidden_step_entropy_mean", "mean"),
        mean_controller_entropy=("controller_entropy_mean", "mean"),
    ).reset_index()

    diversity = per_prompt.groupby("num_rollouts").agg(
        prompts=("example_id", "count"),
        oracle_rate=("any_correct", "mean"),
        mean_unique_answers=("unique_answers", "mean"),
        answer_changed_rate=("unique_answers", lambda values: (values > 1).mean()),
        mean_majority_fraction=("majority_fraction", "mean"),
    ).reset_index()
    display(diversity.style.format({
        "oracle_rate": "{:.2%}",
        "answer_changed_rate": "{:.2%}",
        "mean_majority_fraction": "{:.2%}",
        "mean_unique_answers": "{:.2f}",
    }))

    plot_cols = ["mean_unique_answers", "answer_changed_rate", "mean_majority_fraction"]
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(14, 4))
    for ax, col in zip(axes, plot_cols):
        ax.plot(diversity["num_rollouts"], diversity[col], marker="o")
        ax.set_xscale("log", base=2)
        ax.set_xticks(EXPECTED_ROLLOUTS)
        ax.set_xticklabels(EXPECTED_ROLLOUTS)
        ax.set_title(col)
        ax.set_xlabel("num_rollouts")
        if col.endswith("rate") or col.endswith("fraction"):
            ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.tight_layout()
    plt.show()

## Variant Entropy

Entropy metrics are averaged over the rollouts selected by each variant. This helps check whether extra rollouts improve accuracy by finding lower-uncertainty trajectories or simply by increasing oracle coverage.

In [ ]:
if variants_df.empty:
    display(variants_df)
else:
    variants_df["correct"] = variants_df["correct"].astype(bool)
    metric_cols = [
        "decoded_token_entropy_mean",
        "answer_option_entropy_mean",
        "hidden_step_entropy_mean",
        "controller_entropy_mean",
    ]
    variant_summary = variants_df.groupby(["num_rollouts", "variant"]).agg(
        prompts=("example_id", "count"),
        accuracy=("correct", "mean"),
        **{col: (col, "mean") for col in metric_cols},
    ).reset_index()
    display(variant_summary.style.format({
        "accuracy": "{:.2%}",
        "decoded_token_entropy_mean": "{:.4f}",
        "answer_option_entropy_mean": "{:.4f}",
        "hidden_step_entropy_mean": "{:.4f}",
        "controller_entropy_mean": "{:.4f}",
    }))

    plt.figure(figsize=(9, 5))
    if HAS_SEABORN:
        sns.lineplot(data=variant_summary, x="num_rollouts", y="accuracy", hue="variant", marker="o")
    else:
        for variant, group in variant_summary.groupby("variant"):
            plt.plot(group["num_rollouts"], group["accuracy"], marker="o", label=variant)
        plt.legend()
    plt.xscale("log", base=2)
    plt.xticks(EXPECTED_ROLLOUTS, EXPECTED_ROLLOUTS)
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.xlabel("Number of rollouts per prompt")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Variants Across Rollout Counts")
    plt.tight_layout()
    plt.show()

## Diagnostics

Inspect examples where additional rollouts create disagreement, oracle-only recoveries, or confident incorrect best-of-N decisions.

In [ ]:
if not rollouts_df.empty:
    disagreement = per_prompt.sort_values(["num_rollouts", "unique_answers", "majority_fraction"], ascending=[False, False, True])
    display_cols = ["num_rollouts", "run_id", "example_id", "rollout_count", "unique_answers", "majority_fraction", "any_correct", "random_accuracy_estimate", "mean_hidden_entropy"]
    print("Highest answer disagreement")
    display(disagreement[display_cols].head(30).style.format({
        "majority_fraction": "{:.2%}",
        "random_accuracy_estimate": "{:.2%}",
        "mean_hidden_entropy": "{:.4f}",
    }))

if not variants_df.empty:
    pivot = variants_df.pivot_table(index=["num_rollouts", "run_id", "example_id"], columns="variant", values="correct", aggfunc="first").reset_index()
    oracle_only = pivot[(pivot.get("oracle") == True) & (pivot.get("best_of_n") == False)]
    print(f"Oracle-only recoveries: {len(oracle_only):,}")
    display(oracle_only.sort_values(["num_rollouts", "example_id"]).head(40))

    confident_wrong = variants_df[
        (variants_df["variant"] == "best_of_n")
        & (~variants_df["correct"].astype(bool))
    ].sort_values(["num_rollouts", "hidden_step_entropy_mean"], ascending=[False, True])
    print("Low-hidden-entropy incorrect best-of-N prompts")
    display(confident_wrong[["num_rollouts", "run_id", "example_id", "selected_answer_keys", "gold_answer", "hidden_step_entropy_mean", "decoded_token_entropy_mean"]].head(40))